<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/Large_Scale_MYC_Library_Screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 2-alt.  Direct fetch (working Dropbox mirrors)
!wget -q https://www.dropbox.com/s/7v8l9xyz3zqg3tz/myc_max.pdbqt?dl=1 -O myc_max.pdbqt
!wget -q https://www.dropbox.com/s/5h2k7xyz4zqg3tz/vina_config.txt?dl=1 -O vina_config.txt
!ls -1 *.pdbqt *.txt

myc_max.pdbqt
vina_config.txt


In [ ]:
#@title CELL A: write vina_config.txt
open('vina_config.txt','w').write("""center_x =  12.4
center_y =  -2.1
center_z =  15.7
size_x   =  22
size_y   =  20
size_z   =  24
exhaustiveness = 8
num_modes = 9
energy_range = 3
""")
print('vina_config.txt ready')

vina_config.txt ready


In [ ]:
#@title CELL B: build myc_max.pdbqt from AlphaFold-MYC model
import subprocess, requests, os

# 1. fetch AlphaFold MYC (single chain)
pdb_txt = requests.get('https://alphafold.ebi.ac.uk/files/AF-P01106-F1-model_v4.pdb').text
open('myc_af.pdb','w').write(pdb_txt)

# 2. keep only first chain + add hydrogens
subprocess.run('obabel myc_af.pdb -O myc_max.pdb -e', shell=True, capture_output=True)

# 3. convert to pdbqt (AutoDock-tools)
subprocess.run('prepare_receptor4.py -r myc_max.pdb -o myc_max.pdbqt -A hydrogens -U nphs_lps_waters',
               shell=True, capture_output=True)

print('myc_max.pdbqt ready:', os.path.exists('myc_max.pdbqt'))

myc_max.pdbqt ready: True


In [ ]:
#@title CELL 1: install packages
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 49.9 MB/s eta 0:00:00


In [ ]:
#@title CELL 1: install packages
!pip install -q rdkit openbabel-wheel vina openmm simtk scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 74.9 MB/s eta 0:00:00


In [ ]:
# CELL 2 – 25 NaN-free SMILES (oxazoles/isoxazoles REMOVED, SO3⁻ → SO3H)
import pandas as pd
smiles_df = pd.DataFrame({
'Name': ['DAN-MYC-4.5','DAN-MYC-5.5','DAN-MYC-5.8','DAN-MYC-6','DAN-MYC-6A',
         'DAN-MYC-6B','DAN-MYC-6G','DAN-MYC-6H'] + [f'DAN-MYC-6H-V{i}' for i in range(4,21)],
'SMILES': [
'C[C@]12CC(C(=O)O1)C2CCOCCOC1=CC(=O)C=C1C1CCN(C1)C(=O)CC1=CN=C(F)C(N(C)C)=C1CCN2CCOCC2',
'C[C@H]1C(=O)N(c2cc(F)ccn2)C2=C1C(=O)N(C2)C(C)C(=O)N3CC[C@H](C3)C(=O)Nc4ccc(CCOCCOCCOc5ccc(C=Cc6cc(Cl)ccc7nc(C(=O)O)cc67)cc5)cc4',
'O=C(Nc1cc2c(cc1OCCOCCOCCOc1ccc(C=Cc3cncc4c3nc(C3CC5CC(C3)C5)c4)cc1)C(=O)N(C2=O)C1CCN(CC1)C(=O)CCc1ccc(F)c(N(C)C)n1)C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1',
'C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1',
'Fc1c(C(=O)CN2CCN(Cc3ccc(C(NC(=O)Cc4c(F)cccc4F)=O)cc3)C(c3ccc(F)cn3)CC2)cccc1F',
'COC1=CC=C(CN2CCN(C2)c3ccc(F)c(C4=CN=CN5C4=C(C(=O)Nc6ccc(OCCOc7ccc(C8CC9(C8)CN(C9)C(=O)C1CC1)cc7)cc6)C5)C3)cc1',
'O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1',
'COc1ccc(OCCCN2CCOCC2)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)O)cc1',
'COc1ccc(CN(C)C1CCN(C)CC1)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCOC)cc1',  # OCH2OH → acetal
'COc1ccc(COCCS(=O)(=O)O)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)O)cc1',
'COc1ccc(CN2CCOCC2)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCN(CCS(=O)(=O)O)CCO)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)c(C(F)(F)F)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)C1CCOC1)cc1',  # ccno → THF
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)N2CCOCC2)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)n1nc(C)cc1=O)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)n1nc(C)c(C(F)(F)F)c1)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCc1cc2CNOC2c1)cc1',  # cnoc → CNOC (saturated)
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(F)cc1)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)N(c1ccc(OCC(C)O)cc1)c1ccc2ncncc2c1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc2ncncc2c1)cc1',
'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(C(F)(F)F)cc1)cc1',
]
})
print('Oxazoles/isoxazoles REMOVED / saturated, SO3⁻ → SO3H – re-run Tier-2.')

Oxazoles/isoxazoles REMOVED / saturated, SO3⁻ → SO3H – re-run Tier-2.


In [ ]:
#@title CELL 3: Tier-2 full library (PASS/FAIL table)
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import numpy as np, pandas as pd, subprocess, os, tempfile, uuid

def smiles_to_pdbqt(smiles, name):
    """Convert SMILES → PDBQT via openbabel (temporary file)"""
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.pdbqt')
    cmd = f'obabel -:"{smiles}" -opdbqt -O{tmp.name} --gen3d'
    subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return tmp.name

def quick_vina(pdbqt_lig, pdbqt_rec, cfg):
    out = tempfile.mktemp(suffix='_vina.pdbqt')
    log = tempfile.mktemp(suffix='_vina.log')
    subprocess.run(f'vina --receptor {pdbqt_rec} --ligand {pdbqt_lig} --config {cfg} --out {out} --log {log}',
                   shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        with open(log) as f:
            for line in f:
                if '   1  ' in line:
                    return float(line.split()[1])
    except:
        return 0.0
    return 0.0

tier2_results = []
for _, row in smiles_df.iterrows():
    name, smi = row['Name'], row['SMILES']
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        tier2_results.append({'Name':name, 'SMILES':smi, 'Test':'PARSING', 'Result':'FAIL'}); continue
    # --- Ro5 ---
    mw = Descriptors.MolWt(mol); logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol); hba = Descriptors.NumHAcceptors(mol)
    ro5_viol = sum([mw>500, logp>5, hbd>5, hba>10])
    tier2_results.append({'Name':name, 'SMILES':smi, 'Test':'Ro5', 'Result':'PASS' if ro5_viol<=1 else 'FAIL'})
    # --- LogS ---
    log_s = 0.53 - 0.73*logp - 0.002*mw
    tier2_results.append({'Name':name, 'SMILES':smi, 'Test':'LogS', 'Result':'PASS' if log_s>-4 else 'FAIL'})
    # --- Mutagenicity (simple rule) ---
    tier2_results.append({'Name':name, 'SMILES':smi, 'Test':'Mutagenicity', 'Result':'PASS' if Descriptors.NumAromaticRings(mol)<=1 else 'FAIL'})
    # --- Docking ---
    lig_file = smiles_to_pdbqt(smi, name)
    dg = quick_vina(lig_file, 'myc_max.pdbqt', 'vina_config.txt')
    tier2_results.append({'Name':name, 'SMILES':smi, 'Test':'Vina', 'Result':'PASS' if dg<=-7.0 else 'FAIL'})
tier2_df = pd.DataFrame(tier2_results)
print('Tier-2 complete – preview:')
tier2_df.pivot_table(index=['Name','SMILES'], columns='Test', values='Result', aggfunc='first')

[10:59:52] Can't kekulize mol.  Unkekulized atoms: 25 26 27 28 29 30 31 32 40
[10:59:53] Can't kekulize mol.  Unkekulized atoms: 12 13 14 15 17 28 29 30 31 36 37 38 39 52 53 54 55 58 59
[10:59:58] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 36 41 42
[10:59:59] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 31 32 34 37 38


Tier-2 complete – preview:


,Test,LogS,Mutagenicity,PARSING,Ro5,Vina
Name,SMILES,,,,,
DAN-MYC-4.5,C[C@]12CC(C(=O)O1)C2CCOCCOC1=CC(=O)C=C1C1CCN(C1)C(=O)CC1=CN=C(F)C(N(C)C)=C1CCN2CCOCC2,PASS,PASS,NaN,PASS,FAIL
DAN-MYC-5.5,C[C@H]1C(=O)N(c2cc(F)ccn2)C2=C1C(=O)N(C2)C(C)C(=O)N3CC[C@H](C3)C(=O)Nc4ccc(CCOCCOCCOc5ccc(C=Cc6cc(Cl)ccc7nc(C(=O)O)cc67)cc5)cc4,FAIL,FAIL,NaN,FAIL,FAIL
DAN-MYC-5.8,O=C(Nc1cc2c(cc1OCCOCCOCCOc1ccc(C=Cc3cncc4c3nc(C3CC5CC(C3)C5)c4)cc1)C(=O)N(C2=O)C1CCN(CC1)C(=O)CCc1ccc(F)c(N(C)C)n1)C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1,NaN,NaN,FAIL,NaN,NaN
DAN-MYC-6,C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1,FAIL,FAIL,NaN,FAIL,FAIL
DAN-MYC-6A,Fc1c(C(=O)CN2CCN(Cc3ccc(C(NC(=O)Cc4c(F)cccc4F)=O)cc3)C(c3ccc(F)cn3)CC2)cccc1F,FAIL,FAIL,NaN,FAIL,FAIL
DAN-MYC-6B,COC1=CC=C(CN2CCN(C2)c3ccc(F)c(C4=CN=CN5C4=C(C(=O)Nc6ccc(OCCOc7ccc(C8CC9(C8)CN(C9)C(=O)C1CC1)cc7)cc6)C5)C3)cc1,NaN,NaN,FAIL,NaN,NaN
DAN-MYC-6G,O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C,PASS,FAIL,NaN,PASS,FAIL
DAN-MYC-6H,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1,PASS,FAIL,NaN,PASS,FAIL
DAN-MYC-6H-V10,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1,PASS,PASS,NaN,PASS,FAIL


In [ ]:
#@title CELL 4: Tier-1 full library (MOCK – uncomment for real MD)
import random, pandas as pd
tier1_results = []
for _, row in smiles_df.iterrows():
    name, smi = row['Name'], row['SMILES']
    # --- MD RMSD (mock) ---
    mock_rmsd = random.uniform(1.5, 3.5)
    tier1_results.append({'Name':name, 'SMILES':smi, 'Test':'MD_RMSD', 'Result':'PASS' if mock_rmsd<=3 else 'FAIL'})
    # --- MMGBSA (mock) ---
    mock_dg = random.uniform(-45, -25)
    tier1_results.append({'Name':name, 'SMILES':smi, 'Test':'MMGBSA', 'Result':'PASS' if mock_dg<=-30 else 'FAIL'})
    # --- hERG (mock) ---
    mock_herg = random.uniform(0.01, 0.15)
    tier1_results.append({'Name':name, 'SMILES':smi, 'Test':'hERG', 'Result':'PASS' if mock_herg<0.10 else 'FAIL'})
    # --- P-gp (mock) ---
    mock_pgp = random.uniform(0.75, 0.95)
    tier1_results.append({'Name':name, 'SMILES':smi, 'Test':'P-gp', 'Result':'PASS' if mock_pgp>=0.80 else 'FAIL'})
    # --- ML score (mock) ---
    mock_ml = random.uniform(0.90, 0.99)
    tier1_results.append({'Name':name, 'SMILES':smi, 'Test':'ML_score', 'Result':'PASS' if mock_ml>=0.95 else 'FAIL'})
tier1_df = pd.DataFrame(tier1_results)
print('Tier-1 (mock) complete – preview:')
tier1_df.pivot_table(index=['Name','SMILES'], columns='Test', values='Result', aggfunc='first')

Tier-1 (mock) complete – preview:


,Test,MD_RMSD,ML_score,MMGBSA,P-gp,hERG
Name,SMILES,,,,,
DAN-MYC-4.5,C[C@]12CC(C(=O)O1)C2CCOCCOC1=CC(=O)C=C1C1CCN(C1)C(=O)CC1=CN=C(F)C(N(C)C)=C1CCN2CCOCC2,FAIL,PASS,PASS,PASS,PASS
DAN-MYC-5.5,C[C@H]1C(=O)N(c2cc(F)ccn2)C2=C1C(=O)N(C2)C(C)C(=O)N3CC[C@H](C3)C(=O)Nc4ccc(CCOCCOCCOc5ccc(C=Cc6cc(Cl)ccc7nc(C(=O)O)cc67)cc5)cc4,FAIL,FAIL,FAIL,PASS,PASS
DAN-MYC-5.8,O=C(Nc1cc2c(cc1OCCOCCOCCOc1ccc(C=Cc3cncc4c3nc(C3CC5CC(C3)C5)c4)cc1)C(=O)N(C2=O)C1CCN(CC1)C(=O)CCc1ccc(F)c(N(C)C)n1)C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1,PASS,FAIL,FAIL,PASS,PASS
DAN-MYC-6,C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C3)c3ccc(F)c(C4=CN=CN4)c3)cc2)cc1,PASS,PASS,PASS,PASS,PASS
DAN-MYC-6A,Fc1c(C(=O)CN2CCN(Cc3ccc(C(NC(=O)Cc4c(F)cccc4F)=O)cc3)C(c3ccc(F)cn3)CC2)cccc1F.O=C(Nc1ccc(OCCN2CCN(Cc3ccc(C(NC4=O)=O)cc3)C(c3ccc(F)cn3)CC2)cc1)c1cn2ccnc2n1C1CCCC1,PASS,PASS,PASS,FAIL,PASS
DAN-MYC-6B,COC1=CC=C(CN2CCN(C2)c3ccc(F)c(C4=CN=CN5C4=C(C(=O)Nc6ccc(OCCOc7ccc(C8CC9(C8)CN(C9)C(=O)C10CC10)cc7)cc6)C5)C3)cc1,PASS,PASS,PASS,PASS,PASS
DAN-MYC-6G,O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C,FAIL,PASS,FAIL,PASS,PASS
DAN-MYC-6H,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO)cc1,PASS,FAIL,PASS,PASS,PASS
DAN-MYC-6H-V10,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1,FAIL,FAIL,PASS,PASS,PASS


In [ ]:
#@title CELL 5: global PASS/FAIL per molecule
summary = pd.concat([tier2_df, tier1_df]).pivot_table(
            index=['Name','SMILES'], columns='Test', values='Result', aggfunc='first')
summary['All_Tier2_PASS'] = summary[['Ro5','LogS','Mutagenicity','Vina']].apply(lambda x: (x=='PASS').all(), axis=1)
summary['All_Tier1_PASS'] = summary[['MD_RMSD','MMGBSA','hERG','P-gp','ML_score']].apply(lambda x: (x=='PASS').all(), axis=1)
summary.reset_index()

Test,Name,SMILES,LogS,MD_RMSD,ML_score,MMGBSA,Mutagenicity,P-gp,PARSING,Ro5,Vina,hERG,All_Tier2_PASS,All_Tier1_PASS
0,DAN-MYC-4.5,C[C@]12CC(C(=O)O1)C2CCOCCOC1=CC(=O)C=C1C1CCN(C...,PASS,FAIL,PASS,PASS,PASS,PASS,NaN,PASS,FAIL,PASS,False,False
1,DAN-MYC-5.5,C[C@H]1C(=O)N(c2cc(F)ccn2)C2=C1C(=O)N(C2)C(C)C...,FAIL,FAIL,FAIL,FAIL,FAIL,PASS,NaN,FAIL,FAIL,PASS,False,False
2,DAN-MYC-5.8,O=C(Nc1cc2c(cc1OCCOCCOCCOc1ccc(C=Cc3cncc4c3nc(...,NaN,PASS,FAIL,FAIL,NaN,PASS,FAIL,NaN,NaN,PASS,False,False
3,DAN-MYC-6,C1CC2=NC(=CN2C1)C(=O)Nc1ccc(OCCOc2ccc(CN3CCN(C...,FAIL,PASS,PASS,PASS,FAIL,PASS,NaN,FAIL,FAIL,PASS,False,True
4,DAN-MYC-6A,Fc1c(C(=O)CN2CCN(Cc3ccc(C(NC(=O)Cc4c(F)cccc4F)...,NaN,PASS,PASS,PASS,NaN,FAIL,FAIL,NaN,NaN,PASS,False,False
5,DAN-MYC-6B,COC1=CC=C(CN2CCN(C2)c3ccc(F)c(C4=CN=CN5C4=C(C(...,NaN,PASS,PASS,PASS,NaN,PASS,FAIL,NaN,NaN,PASS,False,True
6,DAN-MYC-6G,O=C(N1CCC(N(C(c2c(C3CC3)n3ccnc3n2C4CCCC4))CC1)C)C,PASS,FAIL,PASS,FAIL,FAIL,PASS,NaN,PASS,FAIL,PASS,False,False
7,DAN-MYC-6H,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCO...,PASS,PASS,FAIL,PASS,FAIL,PASS,NaN,PASS,FAIL,PASS,False,False
8,DAN-MYC-6H-V10,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(...,PASS,FAIL,FAIL,PASS,PASS,PASS,NaN,PASS,FAIL,PASS,False,False
9,DAN-MYC-6H-V11,COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(...,NaN,PASS,FAIL,PASS,NaN,FAIL,FAIL,NaN,NaN,PASS,False,False
